# signal_state_best_n.pt 파인튜닝 (좌회전 데이터 추가)

기존 `red`/`green_straight`/`green_left` 3클래스 모델을 좌회전 데이터로 이어서 학습한다.

**데이터셋 준비 완료(2026-08-23, 2차 갱신):** 새 좌회전 배치 82장(`green_left`) + 기존
3클래스 82장을 병합한 뒤, 첫 재학습이 런타임 끊김으로 `best.pt`를 날려 재현하기 전에
현재 모델을 `좌회전파인튜닝1/2/3`(원본 6349장, 5장 중 1장 샘플링) 전체에 돌려 `green_left`
confidence가 애매하거나 다른 클래스로 오검출된 155장을 뽑았고, 그중 21장을 실제
라벨링(전부 `green_left`로 확인됨)해 추가로 합쳤다. 최종 `signal_state_dataset_0823.zip`:
**train 149장**(red 11/green_straight 35/green_left 103), **valid 36장**(red 5/
green_straight 8/green_left 23). `red`/`green_straight` 표본이 적은 건 `green_left`
보강이 목적인 이번 파인튜닝에서는 의도된 것 — 다만 §검증 단계에서 이 두 클래스 recall이
떨어지지 않았는지 반드시 확인할 것(catastrophic forgetting 위험).

**이번엔 학습 끝나자마자 `best.pt`부터 다운로드할 것** — 지난번엔 export/다운로드 순서를
미루다가 런타임이 끊겨서 그 라운드 결과(pt)를 통째로 날렸다(onnx만 겨우 건짐). §학습 셀
바로 다음에 pt 다운로드 셀을 추가해뒀다.

In [ ]:
!pip install -q ultralytics

## 로컬 파일 올리기

**모델(.pt)과 데이터셋이 지금 로컬 PC에 있는 경우** — Drive에 미리 올려둘 필요 없이
Colab 파일 업로드 창으로 바로 넣을 수 있다. 아래 셀을 실행하면 파일 선택 창이 뜬다.

- `signal_state_best_n.pt` (기존 학습된 체크포인트, 로컬 `yolo_ros/`에 있음)
- `signal_state_dataset_0823.zip` (§0에서 이미 병합/분할까지 끝내둔 데이터셋 —
  train 149장 + valid 36장, `data.yaml` 포함)

두 파일 다 크지 않아서(.pt 수 MB, 데이터셋 zip 수십 MB 수준) Drive 안 거치고 바로
업로드하는 게 더 빠르다. 세션이 끊기면(런타임 재시작 등) 업로드한 파일도 날아가니
그때는 이 셀부터 다시 실행.

In [ ]:
from google.colab import files

print('signal_state_best_n.pt 선택')
uploaded_ckpt = files.upload()
BASE_CKPT = list(uploaded_ckpt.keys())[0]
print('업로드됨:', BASE_CKPT)

In [ ]:
print('데이터셋 zip 선택 (Roboflow export 등)')
uploaded_data = files.upload()
DATA_ZIP = list(uploaded_data.keys())[0]

!unzip -q -o "$DATA_ZIP" -d dataset
!ls dataset

DATA_YAML = 'dataset/data.yaml'  # unzip 결과 구조가 다르면 실제 경로로 수정할 것

### (대안) Drive를 쓰는 경우

매번 업로드하기 귀찮으면 Drive에 한 번 올려두고 마운트해서 쓰는 방법도 가능 — 이 경우
위 두 업로드 셀은 건너뛰고 아래 셀만 실행:

```python
from google.colab import drive
drive.mount('/content/drive')

BASE_CKPT = '/content/drive/MyDrive/UMK/yolo_ros/signal_state_best_n.pt'
DATA_YAML = '/content/drive/MyDrive/UMK/datasets/signal_state/data.yaml'
```

## 이어서 학습 (파인튜닝)

`yolov8n.pt`(COCO 베이스)가 아니라 **방금 업로드한, 이미 학습된 `signal_state_best_n.pt`**를
불러와서 바로 `.train()`을 부르면 그게 곧 이어서 학습(파인튜닝)이다 — `red`/`green_straight`를
이미 아는 가중치에서 시작하므로 그 두 클래스를 새로 배우는 게 아니라 유지한 채
`green_left`만 보강된다.

(`resume=True`는 여기 쓰는 게 아님 — 그건 중단된 학습을 같은 run에서 이어갈 때 쓰는
다른 옵션. 지금처럼 "이미 끝난 모델을 새 데이터로 더 학습"할 땐 그냥 아래처럼 한다.)

In [ ]:
from ultralytics import YOLO

model = YOLO(BASE_CKPT)

results = model.train(
    data=DATA_YAML,
    epochs=30,            # 처음 학습보다 짧게 — 이미 수렴한 모델의 미세조정
    imgsz=640,
    batch=16,
    lr0=0.001,             # 기본값(0.01)보다 낮춰 기존 클래스 붕괴 방지
    patience=10,           # val 성능 정체 시 조기종료
    project='signal_state_finetune',
    name='left_boost_0823',
    exist_ok=True,
)

## ⚠ 학습 끝나자마자 바로 실행 — pt부터 챙기기

지난 라운드에서 이 단계를 건너뛰고 검증/export로 넘어갔다가 런타임이 끊겨 `best.pt`를
통째로 날린 적이 있다(onnx만 겨우 복구). 검증/export보다 먼저 이 셀부터 실행한다.

In [ ]:
from google.colab import files

best_pt = model.trainer.best   # runs/.../weights/best.pt
print(best_pt)
files.download(str(best_pt))

# Drive에도 백업 — 로컬 다운로드까지 실패해도 여기 남아있음
from google.colab import drive
drive.mount('/content/drive')

import shutil, os
os.makedirs('/content/drive/MyDrive/UMK_backup', exist_ok=True)
shutil.copy(str(best_pt), '/content/drive/MyDrive/UMK_backup/signal_state_best_n_0823.pt')

## 검증 — 클래스별 성능 확인

`green_left`만 오르고 `red`/`green_straight`가 떨어졌으면 lr을 더 낮추거나 epoch을
줄여 재시도할 것.

In [ ]:
metrics = model.val(data=DATA_YAML)
print(metrics.box.maps)   # 클래스별 mAP50-95: [red, green_straight, green_left] 순

# runs/detect/.../confusion_matrix.png 도 열어서 green_left가 실제로 green_straight나
# 배경과 헷갈리고 있는지 눈으로 확인할 것

## ONNX export (기존 규약 그대로: imgsz=640, opset=12, simplify=True, nms=True)

In [ ]:
best_pt = model.trainer.best   # runs/.../weights/best.pt
best_model = YOLO(best_pt)
best_model.export(format='onnx', imgsz=640, opset=12, simplify=True, nms=True)

## 결과 파일 로컬로 다운로드

업로드 방식으로 진행했다면(Drive 안 씀) 결과 `.onnx`를 로컬 PC로 바로 다운로드한다.

In [ ]:
from google.colab import files

onnx_path = str(best_pt).replace('.pt', '.onnx')
files.download(onnx_path)

## 마지막

다운로드된 새 `signal_state_best_n.onnx`를 저장소 `yolo_ros/signal_state_best_n.onnx`에
덮어쓰고, 실차에서 `DEBUG_VIZ_YOLO_SIGNAL_STATE=True`로 기존 Hough 판정과 비교 검증할 것.
실차 미검증 상태로는 `perc_signal()` 판단 소스로 바꾸지 않는다(저장소 관례).